# Notebook 07 — Python Environment and Packaging Basics

**Module:** 00 — Orientation  
**Notebook:** 07 of 13  
**Prerequisites:** Notebooks 01–06  
**Time estimate:** 60 minutes

This notebook explains virtual environments, the three dependency file formats (`requirements.txt`, `environment.yml`, `pyproject.toml`), and performs an editable install of `compbio_utils`.

---
## Step 1 — Motivation

Without environment isolation, installing one project's dependencies can silently break another project. This is the "works on my machine" problem at the package level.

Virtual environments guarantee isolation: each project sees only the packages installed into its own environment. The `requirements.txt` / `environment.yml` / `pyproject.toml` files guarantee reproducibility: anyone can recreate the exact same environment from the file.

For Track A/B: a repository that cannot be bootstrapped from a clean clone is not a portfolio artifact — it is a local script collection.

---
## Step 2 — Intuition

| Analogy | Environment concept |
|---------|--------------------|
| A separate locker for each project | Virtual environment (venv / conda env) |
| A packing list of what's in the locker | `requirements.txt` |
| A more detailed packing list with exact versions | `requirements.txt` pinned with `==` |
| A build spec for the locker itself | `pyproject.toml` |

**`pip install -e .`** (editable install) means: "install this package from this folder, but don't copy it — instead, any change to the source files is immediately reflected when you import the package." This is what you use for `compbio_utils` so that every module can import from it without re-installing after each edit.

---
## Step 3 — Biological Background

*Not applicable.*

---
## Step 4 — Mathematical Explanation

*Not applicable.*

---
## Step 5 — Computational Explanation

**The three dependency file formats — when to use each:**

| File | Tool | Use case |
|------|------|----------|
| `requirements.txt` | pip | Simple list of packages for a project or environment |
| `environment.yml` | conda | Full conda environment including non-Python dependencies |
| `pyproject.toml` | pip / build | Package metadata + dependencies for a *distributable package* |

This program uses `requirements.txt` for the repository environment and `pyproject.toml` for `compbio_utils` (the installable package). `environment.yml` is provided as an alternative for conda users (see `ENVIRONMENT.md`).

**`uv` vs `conda`:**  
- `uv` is faster and pip-compatible. Recommended for this program.  
- `conda` handles non-Python dependencies (e.g., BLAST, samtools) better.  
- Module 09 (NGS) will use conda-installed bioinformatics tools — revisit the choice there.

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Show the active environment
import sys
import subprocess

print(f"Python executable: {sys.executable}")
print(f"Python version:    {sys.version.split()[0]}")
print()

# Is this a virtual environment?
import pathlib
venv_marker = pathlib.Path(sys.prefix) / "pyvenv.cfg"
if venv_marker.exists():
    print(f"Virtual environment: YES")
    print(f"  Location: {sys.prefix}")
else:
    print("Virtual environment: NOT DETECTED — are you using the system Python?")
    print("  Recommended: create a virtual environment per ENVIRONMENT.md")

In [ ]:
# Cell 6.2 — List installed packages (top 20 by name)
result = subprocess.run([sys.executable, "-m", "pip", "list", "--format=columns"],
                        capture_output=True, text=True)
lines = result.stdout.strip().splitlines()
# Print header + first 25 packages
for line in lines[:27]:
    print(line)
if len(lines) > 27:
    print(f"... ({len(lines) - 2} packages total)")

In [ ]:
# Cell 6.3 — Check requirements.txt exists and has content
repo_root = pathlib.Path.cwd()
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists():
        break
    repo_root = repo_root.parent

req_file = repo_root / "requirements.txt"
if req_file.exists():
    content = req_file.read_text(encoding="utf-8")
    lines = [l for l in content.splitlines() if l.strip() and not l.startswith("#")]
    print(f"requirements.txt: {len(lines)} packages listed")
    for line in lines[:15]:
        print(f"  {line}")
    if len(lines) > 15:
        print(f"  ... ({len(lines) - 15} more)")
else:
    print("requirements.txt not found at repo root.")
    print("Create it: pip freeze > requirements.txt")

In [ ]:
# Cell 6.4 — Install compbio_utils in editable mode and verify
compbio_path = repo_root / "utilities" / "compbio_utils"

if compbio_path.exists():
    print(f"compbio_utils path: {compbio_path}")
    # Try importing
    try:
        import compbio_utils
        print(f"✓ already importable — version: {compbio_utils.__version__}")
    except ImportError:
        print("Not yet installed. Run in terminal:")
        print(f"  pip install -e {compbio_path}")
        print("Then re-run this cell.")
else:
    print("utilities/compbio_utils not found.")
    print("Run Notebook 02 (Environment Setup) to seed the package skeleton.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Environment isolation diagram
diagram = """
  Environment Isolation
  =====================

  System Python (don't touch)       venv: computational-biology
  ─────────────────────────         ─────────────────────────────
  Python 3.x (system)               Python 3.12
  various system packages            numpy ≥ 1.26
  (possibly outdated)                scipy ≥ 1.12
                                     pandas ≥ 2.2
                                     biopython ≥ 1.83
                                     compbio_utils (editable)
                                     ... (from requirements.txt)

  Activation: source venv/bin/activate  (Linux/Mac)
              venv\\Scripts\\activate     (Windows)
              (or: uv sync, which handles this automatically)
"""
print(diagram)

---
## Step 8 — Exercises

**Exercise 1:**  
Run `pip uninstall seaborn -y` in your terminal. Re-run Cell 6.2. Confirm seaborn is gone.  
Re-install: `pip install seaborn`. Confirm it returns.

**Exercise 2:**  
Run `pip install -e utilities/compbio_utils` if not already done. Confirm `import compbio_utils` works in Cell 6.4. What is the version string?

**Exercise 3:**  
In your own words: what is the difference between `pip install seaborn` and `pip install -e utilities/compbio_utils`? Write the answer in `exercises/07_packaging_exercises.md`.

---
## Quiz — Active Recall

1. What does the `-e` flag mean in `pip install -e .`?
2. When would you use `environment.yml` instead of `requirements.txt`?
3. What is the difference between `requirements.txt` and `pyproject.toml`?
4. If you edit `compbio_utils/sequence.py` after an editable install, do you need to re-install? Why?
5. How do you verify which Python executable is being used inside a running notebook?

---
## Reflection

**Date completed:** ____________________

**Reflection:**

> *[Is compbio_utils installable? What does editable install mean in practice? What is one dependency management decision you'd make differently after this notebook?]*

---
## References

- [uv documentation](https://docs.astral.sh/uv/)
- [pip documentation](https://pip.pypa.io/en/stable/)
- [pyproject.toml specification](https://packaging.python.org/en/latest/guides/writing-pyproject-toml/)
- `ENVIRONMENT.md` — primary reference for this program's environment setup

---
*Next notebook: `07_scientific_computing_workflow.ipynb`*